# Phase 1 — Data Loading & Exploration

**Goal:** Load the NSL-KDD dataset, understand its structure, and visualize key patterns.

The NSL-KDD dataset contains network connection records. Each row represents one connection
(e.g., a TCP session) and has 41 features describing it, plus a label (normal or attack type).

**Before running this notebook:**
1. Download the NSL-KDD dataset from https://www.unb.ca/cic/datasets/nsl.html or Kaggle
2. Place `KDDTrain+.txt` and `KDDTest+.txt` in the `data/` folder

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look nicer
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Libraries loaded successfully!")

## 2. Define Column Names

The NSL-KDD `.txt` files have **no headers**, so we need to define them ourselves.
These 41 feature names come from the original KDD Cup 1999 documentation.

The features fall into 3 categories:
- **Basic features** (1-9): duration, protocol, bytes, etc.
- **Content features** (10-22): failed logins, shells, etc.
- **Traffic features** (23-41): connection rates, error rates, etc.

In [ ]:
# All 41 feature column names from the KDD dataset documentation
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label",          # attack type or 'normal'
    "difficulty_level" # not used for training, ignore this
]

print(f"Total columns defined: {len(columns)}")

## 3. Load the Dataset

We use `pd.read_csv()` with `header=None` because the files have no header row,
then assign our column names.

In [ ]:
# Load training and test sets
train_df = pd.read_csv("../data/KDDTrain+.txt", header=None, names=columns)
test_df = pd.read_csv("../data/KDDTest+.txt", header=None, names=columns)

print(f"Training set: {train_df.shape[0]:,} rows, {train_df.shape[1]} columns")
print(f"Test set:     {test_df.shape[0]:,} rows, {test_df.shape[1]} columns")

## 4. First Look at the Data

Let's see what the data actually looks like — the first few rows, data types, and basic stats.

In [ ]:
# Show the first 5 rows
train_df.head()

In [ ]:
# Data types: most are numbers (int64/float64), a few are text (object)
# The text columns are: protocol_type, service, flag, label
train_df.dtypes

In [ ]:
# Quick statistical summary of numeric columns
train_df.describe()

In [ ]:
# Check for missing values (NSL-KDD should have none)
missing = train_df.isnull().sum()
print(f"Total missing values: {missing.sum()}")

# If there were missing values, this would show which columns:
# missing[missing > 0]

## 5. Explore the Categorical Columns

There are 3 categorical (text) features in the dataset:
- **protocol_type**: TCP, UDP, or ICMP
- **service**: The network service (http, ftp, smtp, etc.)
- **flag**: Connection status flag (SF = normal, REJ = rejected, etc.)

In [ ]:
# How many unique values does each categorical column have?
for col in ["protocol_type", "service", "flag"]:
    unique_vals = train_df[col].nunique()
    print(f"{col}: {unique_vals} unique values")
    print(f"  Values: {train_df[col].unique()[:10]}")
    print()

## 6. Explore Attack Labels

The `label` column contains the attack type. There are 40+ specific attack names,
but they group into 5 categories:

| Category | What it means | Examples |
|----------|--------------|----------|
| **Normal** | Legitimate traffic | normal |
| **DoS** | Denial of Service — flood the target | neptune, smurf, back |
| **Probe** | Scanning/reconnaissance | portsweep, nmap, satan |
| **R2L** | Remote to Local — unauthorized remote access | warezclient, spy, ftp_write |
| **U2R** | User to Root — privilege escalation | rootkit, buffer_overflow |

In [ ]:
# How many unique attack labels are there?
print(f"Unique labels in training set: {train_df['label'].nunique()}")
print(f"Unique labels in test set:     {test_df['label'].nunique()}")
print()

# Show all label counts, sorted
print("--- Training set label distribution ---")
print(train_df["label"].value_counts())

In [ ]:
# Map each specific attack to its category (DoS, Probe, R2L, U2R)
# This mapping comes from the NSL-KDD documentation

attack_map = {
    "normal": "Normal",
    # DoS attacks
    "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS",
    "smurf": "DoS", "teardrop": "DoS", "mailbomb": "DoS",
    "apache2": "DoS", "processtable": "DoS", "udpstorm": "DoS",
    # Probe attacks
    "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe",
    "satan": "Probe", "mscan": "Probe", "saint": "Probe",
    # R2L attacks (Remote to Local)
    "ftp_write": "R2L", "guess_passwd": "R2L", "imap": "R2L",
    "multihop": "R2L", "phf": "R2L", "spy": "R2L",
    "warezclient": "R2L", "warezmaster": "R2L", "sendmail": "R2L",
    "named": "R2L", "snmpgetattack": "R2L", "snmpguess": "R2L",
    "xlock": "R2L", "xsnoop": "R2L", "worm": "R2L",
    # U2R attacks (User to Root)
    "buffer_overflow": "U2R", "loadmodule": "U2R", "perl": "U2R",
    "rootkit": "U2R", "httptunnel": "U2R", "ps": "U2R",
    "sqlattack": "U2R", "xterm": "U2R",
}

# Create a new column with the 5-class category
train_df["attack_category"] = train_df["label"].map(attack_map)
test_df["attack_category"] = test_df["label"].map(attack_map)

# Check if any labels didn't get mapped (would show as NaN)
unmapped_train = train_df[train_df["attack_category"].isna()]["label"].unique()
unmapped_test = test_df[test_df["attack_category"].isna()]["label"].unique()

if len(unmapped_train) > 0:
    print(f"WARNING: Unmapped labels in train: {unmapped_train}")
if len(unmapped_test) > 0:
    print(f"WARNING: Unmapped labels in test: {unmapped_test}")
else:
    print("All labels mapped successfully!")

In [ ]:
# Show the 5-class distribution
print("--- Training set: Attack Category Counts ---")
category_counts = train_df["attack_category"].value_counts()
print(category_counts)
print()
print("--- As percentages ---")
print((category_counts / len(train_df) * 100).round(2))

## 7. Visualizations

Now let's make some charts to see what's going on in the data.

### 7a. Attack Category Distribution (Bar Chart)

This shows the **class imbalance** problem — some categories have way more samples than others.
We'll deal with this in Phase 4 using SMOTE.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of counts
category_order = ["Normal", "DoS", "Probe", "R2L", "U2R"]
colors = ["#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#e67e22"]

sns.countplot(
    data=train_df, x="attack_category",
    order=category_order, palette=colors, ax=axes[0]
)
axes[0].set_title("Attack Category Counts (Training Set)")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Count")

# Add count labels on top of each bar
for bar in axes[0].patches:
    axes[0].annotate(
        f"{int(bar.get_height()):,}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom", fontsize=10
    )

# Pie chart of percentages
category_counts_ordered = train_df["attack_category"].value_counts().reindex(category_order)
axes[1].pie(
    category_counts_ordered, labels=category_order, colors=colors,
    autopct="%1.1f%%", startangle=90
)
axes[1].set_title("Attack Category Distribution")

plt.tight_layout()
plt.show()

### 7b. Protocol Type Distribution

Most network traffic uses TCP. Let's see how protocols relate to attack types.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall protocol distribution
sns.countplot(data=train_df, x="protocol_type", palette="Set2", ax=axes[0])
axes[0].set_title("Protocol Type Distribution")

# Protocol type broken down by attack category
protocol_attack = pd.crosstab(train_df["protocol_type"], train_df["attack_category"])
protocol_attack[category_order].plot(kind="bar", stacked=True, ax=axes[1], color=colors)
axes[1].set_title("Protocol Type by Attack Category")
axes[1].set_xlabel("Protocol")
axes[1].set_ylabel("Count")
axes[1].legend(title="Category")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### 7c. Top 15 Network Services

The `service` column tells us what application-layer service was used (HTTP, FTP, etc.).

In [ ]:
top_services = train_df["service"].value_counts().head(15)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_services.index, y=top_services.values, palette="viridis")
plt.title("Top 15 Network Services")
plt.xlabel("Service")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 7d. Feature Correlation Heatmap

This shows which numeric features are correlated with each other.
Highly correlated features (dark red/blue) carry similar information —
we might drop some later to simplify the model.

In [ ]:
# Select only numeric columns for correlation
numeric_df = train_df.select_dtypes(include=[np.number])

# Compute correlation matrix
corr_matrix = numeric_df.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(
    corr_matrix, cmap="RdBu_r", center=0,
    square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

### 7e. Duration & Bytes by Attack Category

Let's see if attacks look different from normal traffic in terms of connection duration and data volume.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Duration (capped at 99th percentile to avoid extreme outliers squishing the plot)
cap = train_df["duration"].quantile(0.99)
sns.boxplot(
    data=train_df[train_df["duration"] <= cap],
    x="attack_category", y="duration",
    order=category_order, palette=colors, ax=axes[0]
)
axes[0].set_title("Connection Duration by Category")

# Source bytes
cap = train_df["src_bytes"].quantile(0.99)
sns.boxplot(
    data=train_df[train_df["src_bytes"] <= cap],
    x="attack_category", y="src_bytes",
    order=category_order, palette=colors, ax=axes[1]
)
axes[1].set_title("Source Bytes by Category")

# Destination bytes
cap = train_df["dst_bytes"].quantile(0.99)
sns.boxplot(
    data=train_df[train_df["dst_bytes"] <= cap],
    x="attack_category", y="dst_bytes",
    order=category_order, palette=colors, ax=axes[2]
)
axes[2].set_title("Destination Bytes by Category")

plt.tight_layout()
plt.show()

## 8. Key Takeaways

After exploring the data, here's what we know:

1. **Dataset size:** ~125K training records, ~22K test records, 41 features each
2. **No missing values** — the data is clean
3. **Class imbalance:** Normal and DoS dominate; R2L and U2R have very few samples
   - This will cause the model to be bad at detecting rare attacks
   - We'll fix this with SMOTE in Phase 4
4. **3 categorical features** need encoding (protocol_type, service, flag)
5. **Attack mapping:** 40+ specific attacks map into 5 clean categories

**Next up: Phase 2** — We'll preprocess this data (encode categoricals, normalize numbers) to make it ready for machine learning.